# Exp13: Gemma-2 IOI full run (2B/9B/27B), confound-controlled
Changes vs Exp12:
1. **Equal name counts** per language (Exp12 had EN 45 / FR 34 / YO 20, and the gradient subspace is dominated by name directions, so effective dimension tracked name count).
2. **Same-names variants**: `FRs` / `YOs` = French / Yoruba templates with the *English* names. Separates lexical disjointness from language.
3. **Control subspaces**: random, and **PCA** (top-variance directions of the target's END residual, the energy-matched control for the norm effect behind retention > 1).
4. **Held-out evaluation**: subspaces and head rankings come from estimation prompts (seed 0), effects are measured on a separate eval set. Subspaces are re-estimated per seed.
5. Gate is **per source**; matrices always print with flags. Oracle dropped.
6. Cross-model summary at the end (the scale trend).

Read: cross/own ratio is only meaningful where the target's own subspace passes (own effect > 0.2).
- EN->FR vs EN->FRs: if both are similar, language (not names) drives the transfer pattern.
- Ratio trend over 2B -> 9B -> 27B is the scale result.

Run **SMOKE=True** first. Full run on 27B is long (head scans x3 languages + ~420 erasure evals); results cache per model.

In [1]:
import os
os.environ["http_proxy"]  = "http://proxy61.iitd.ac.in:3128"
os.environ["https_proxy"] = "http://proxy61.iitd.ac.in:3128"
os.environ["HF_TOKEN"] = os.environ.get("HF_TOKEN", "<REDACTED>")


In [2]:
import os
GPU_ID = "0"   # your sentinel; None = all visible GPUs
if GPU_ID is not None: os.environ["CUDA_VISIBLE_DEVICES"] = GPU_ID   # before torch import
import json, gc, random
import numpy as np, pandas as pd, torch
from transformers import AutoTokenizer, AutoModelForCausalLM

SMOKE = False          # True: 2B only, tiny. Run first.
HF_TOKEN = os.environ.get("HF_TOKEN")
os.environ["HF_HUB_OFFLINE"] = "1"      # first line of the first cell, before any hub calls
MODELS = ["/home/models/gemma-2-9b-it", "/home/models/gemma-2-27b-it",
          "google/gemma-2-2b", "/home/models/gemma-2-9b"]
LAYER_FRACS = [0.33, 0.5, 0.67, 0.85]
GRAD_FRACS = [0.33, 0.5, 0.67, 0.85]
CAUSAL_FRAC = 0.5                       # Lh; must be in LAYER_FRACS and GRAD_FRACS
ERASE_MODES = ["all", "end"]
K_ANGLE = 4
GATE_ACC = 0.75
OWN_MIN = 0.2         # target's own-subspace effect must exceed this for a ratio to count
RESULTS_DIR = "exp13_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

if SMOKE:
    MODELS = MODELS[:1]; N_PROMPTS = 60; N_EVAL = 60; SEEDS = [0]; K_HEADS = 5; SCAN_PER_T = 2; K_LIST = [4, 16]
else:
    N_PROMPTS = 420; N_EVAL = 198; SEEDS = [0, 1, 2]; K_HEADS = 10; SCAN_PER_T = 4; K_LIST = [4, 16]
assert CAUSAL_FRAC in LAYER_FRACS and CAUSAL_FRAC in GRAD_FRACS
assert torch.cuda.is_available(), "no CUDA visible: check GPU_ID"
props = torch.cuda.get_device_properties(0)
print("SMOKE" if SMOKE else "FULL", MODELS, torch.cuda.device_count(), "GPU(s):", props.name, round(props.total_memory/2**30), "GiB")
print("27B bf16 needs ~54 GiB + activations (grad pass adds more); 9B ~18; 2B ~5")

/home/anwoy/miniconda3/envs/ke/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


FULL ['/home/models/gemma-2-9b-it', '/home/models/gemma-2-27b-it', 'google/gemma-2-2b', '/home/models/gemma-2-9b'] 1 GPU(s): NVIDIA H200 NVL 140 GiB
27B bf16 needs ~54 GiB + activations (grad pass adds more); 9B ~18; 2B ~5


In [3]:
TEMPLATES = {
 "en": ["When {N1} and {N2} went to the store, {S} gave a drink to",
        "Then, {N1} and {N2} had a lot of fun at the school. {S} gave a ball to",
        "After {N1} and {N2} went to the park, {S} handed a book to",
        "Yesterday {N1} and {N2} visited the office, and {S} sent a letter to",
        "While {N1} and {N2} were at the market, {S} offered some fruit to",
        "Today {N1} and {N2} met in the garden. Afterwards, {S} showed the photos to"],
 "fr": ["Quand {N1} et {N2} sont allés au marché, {S} a donné une pomme à",
        "Ensuite, {N1} et {N2} se sont bien amusés à l'école. {S} a donné un ballon à",
        "Après que {N1} et {N2} sont allés au parc, {S} a prêté un livre à",
        "Hier, {N1} et {N2} ont visité le bureau, et {S} a envoyé une lettre à",
        "Pendant que {N1} et {N2} étaient au marché, {S} a offert des fruits à",
        "Aujourd'hui, {N1} et {N2} se sont retrouvés dans le jardin. Ensuite, {S} a montré les photos à"],
 "yo": ["Nígbà tí {N1} àti {N2} lọ sí ọjà, {S} ra ẹ̀bùn kan fún",
        "Lẹ́yìn náà, {N1} àti {N2} gbádùn ní ilé-ẹ̀kọ́. {S} fi bọ́ọ̀lù kan fún",
        "Lẹ́yìn tí {N1} àti {N2} lọ sí ọgbà, {S} yá ìwé kan fún",
        "Lánàá {N1} àti {N2} ṣàbẹ̀wò sí ọ́fíìsì, {S} sì fi lẹ́tà kan ranṣẹ́ sí",
        "Nígbà tí {N1} àti {N2} wà ní ọjà, {S} mú èso wá fún",
        "Lónìí {N1} àti {N2} pàdé ní ọgbà. Lẹ́yìn náà, {S} fi àwòrán hàn fún"],
}
NAMES = {
 "en": "James John Robert Michael William David Richard Thomas Mary Patricia Jennifer Linda Elizabeth Barbara Susan Jessica Sarah Karen Daniel Mark Paul Steven Kevin Brian George Anna Laura Emma Alice Julia Peter Henry Frank Kate Lucy Ruth Helen Carol Diane Grace Rose Edward Andrew Joseph Charles".split(),
 "fr": "Jean Pierre Louis Jacques Michel Henri Marie Claire Sophie Camille Lucie Chloé Hugo Lucas Antoine Nicolas Alain Luc Éric Anne Isabelle Nathalie Sylvie Céline Julien Mathieu Olivier Bruno Léa Manon Étienne Gérard Vincent Émile Alexandre Margot Inès Mathilde Clémence Aurélie Baptiste Guillaume".split(),
 "yo": "Ayo Tunde Femi Bola Kemi Dayo Tola Segun Yemi Wale Funmi Ade Bisi Folake Kunle Lola Sade Tayo Seyi Bimpe Dupe Yinka Kola Ladi Remi Toyin Jide Gbenga Ife Tobi Temi Wumi Niyi Shola Sola Lekan Dele Kayode Bukola Ronke Shade Bayo Biodun Taiwo Kehinde Lanre".split(),
}
NAMES["fr"] = [n for n in NAMES["fr"] if n not in set(NAMES["en"])]   # keep lexically disjoint from EN

In [4]:
NAMES["yo"] = list(dict.fromkeys(NAMES["yo"] + (
 "Abiola Adebayo Adewale Akin Akinwale Ayodele Babajide Babatunde Bolu Busola Damilola Debo Deji Dotun Efe Eniola "
 "Folu Gbemi Idowu Ige Jumoke Kike Lade Mide Muyiwa Nike Niran Olu Omolara Rotimi Sanmi Simi Tade Tope Tosin Wunmi Yetunde Yomi "
 "Adeola Femisoye Ibukun Jola Lara Moji Ola Tomi Yewande Zainab").split()))

In [5]:
def single_token_names(tok, names):
    out, nid, seen = [], {}, set()
    for n in names:
        ids = tok.encode("a " + n, add_special_tokens=False)
        if len(ids) == 2 and ids[1] not in seen:
            out.append(n); nid[n] = ids[1]; seen.add(ids[1])
    return out, nid

def build(tok, lang, tmpl_ids, names, nid, n, rng):
    per = max(1, n // len(tmpl_ids)); batches = []
    for t in tmpl_ids:
        rows = []
        for _ in range(per):
            io, s = rng.sample(names, 2)
            n1, n2 = (io, s) if rng.random() < 0.5 else (s, io)
            ids = tok(TEMPLATES[lang][t].format(N1=n1, N2=n2, S=s))["input_ids"]
            rows.append((ids, nid[io], nid[s]))
        lens = [len(r[0]) for r in rows]
        L = max(set(lens), key=lens.count)
        rows = [r for r in rows if len(r[0]) == L]
        batches.append(dict(
            x=torch.tensor([r[0] for r in rows]),
            io=torch.tensor([r[1] for r in rows]), s=torch.tensor([r[2] for r in rows]),
            pIO=torch.tensor([r[0].index(r[1]) for r in rows]),
            pS2=torch.tensor([len(r[0]) - 1 - r[0][::-1].index(r[2]) for r in rows])))
    return batches

def slice_b(b, n): return {k: v[:n] for k, v in b.items()}

class Hooks:
    def __init__(self): self.h = []
    def add(self, m, kind, fn): self.h.append(getattr(m, kind)(fn))
    def __enter__(self): return self
    def __exit__(self, *a): [h.remove() for h in self.h]

def subspace(X, k):
    Xc = X - X.mean(0)
    return torch.linalg.svd(Xc, full_matrices=False)[2][:k]

def angle(X, Y, k):
    s = torch.linalg.svdvals(subspace(X, k) @ subspace(Y, k).T).clamp(-1, 1)
    return torch.rad2deg(torch.arccos(s)).mean().item()

In [6]:
@torch.no_grad()
def fwd(model, b, hs_layers=(), bs=32):
    dev = model.model.embed_tokens.weight.device
    cap = getattr(model.config, "final_logit_softcapping", None)
    lds, accs = [], []
    hs = {l: {"IO": [], "S2": [], "END": []} for l in hs_layers}
    for i in range(0, len(b["x"]), bs):
        x = b["x"][i:i+bs].to(dev)
        out = model.model(input_ids=x, output_hidden_states=bool(hs_layers))
        lg = model.lm_head(out.last_hidden_state[:, -1])
        if cap: lg = torch.tanh(lg / cap) * cap
        lg = lg.float(); ar = torch.arange(len(x), device=lg.device)
        d = lg[ar, b["io"][i:i+bs].to(lg.device)] - lg[ar, b["s"][i:i+bs].to(lg.device)]
        lds.append(d.cpu()); accs.append((d > 0).float().cpu())
        for l in hs_layers:
            h = out.hidden_states[l]; a2 = torch.arange(len(x), device=h.device)
            hs[l]["IO"].append(h[a2, b["pIO"][i:i+bs].to(h.device)].float().cpu())
            hs[l]["S2"].append(h[a2, b["pS2"][i:i+bs].to(h.device)].float().cpu())
            hs[l]["END"].append(h[:, -1].float().cpu())
    return torch.cat(lds), torch.cat(accs), {l: {p: torch.cat(v) for p, v in d.items()} for l, d in hs.items()}

def ev(model, batches, hs_layers=()):
    rs = [fwd(model, b, hs_layers) for b in batches]
    ld = torch.cat([r[0] for r in rs]); acc = torch.cat([r[1] for r in rs])
    hs = {l: {p: torch.cat([r[2][l][p] for r in rs]) for p in ("IO", "S2", "END")} for l in hs_layers}
    return ld, acc, hs

def head_hook(hs_idx, HD):
    def fn(mod, args):
        x = args[0].clone()
        for h in hs_idx: x[..., h*HD:(h+1)*HD] = 0
        return (x,) + tuple(args[1:])
    return fn

def ev_heads(model, batches, heads):   # heads: list of (layer, head)
    HD = model.config.head_dim; by = {}
    for l, h in heads: by.setdefault(l, []).append(h)
    with Hooks() as H:
        for l, hh in by.items():
            H.add(model.model.layers[l].self_attn.o_proj, "register_forward_pre_hook", head_hook(hh, HD))
        return ev(model, batches)[0]

def sub_hook(V, mu):
    def fn(mod, args, out):
        tup = isinstance(out, tuple); h = (out[0] if tup else out).clone()
        last = h[:, -1].float(); Vd = V.to(h.device); m = mu.to(h.device)
        h[:, -1] = (last - ((last - m) @ Vd.T) @ Vd).to(h.dtype)
        return (h,) + tuple(out[1:]) if tup else h
    return fn

def ev_sub(model, batches, layer_idx, V, mu):
    with Hooks() as H:
        H.add(model.model.layers[layer_idx], "register_forward_hook", sub_hook(V, mu))
        return ev(model, batches)[0]

In [7]:
def ev_hs(model, batches, layers):
    rs = [fwd(model, b, layers) for b in batches]
    hs = {l: {p: torch.cat([r[2][l][p] for r in rs]) for p in ("IO", "S2", "END")} for l in layers}
    tid = torch.cat([torch.full((len(b["x"]),), t) for t, b in enumerate(batches)])
    return hs, tid

def grad_END(model, b_list, layers, bs=16):
    """per-prompt d(logit_diff)/d(resid at END) for each layer in `layers` (hidden_states index)."""
    dev = model.model.embed_tokens.weight.device
    cap = getattr(model.config, "final_logit_softcapping", None)
    G = {l: [] for l in layers}; tid = []
    for t, b in enumerate(b_list):
        for i in range(0, len(b["x"]), bs):
            x = b["x"][i:i+bs].to(dev)
            with torch.no_grad(): emb = model.model.embed_tokens(x)
            emb = emb.detach().requires_grad_(True)
            with torch.enable_grad():
                out = model.model(inputs_embeds=emb, output_hidden_states=True)
                for l in layers: out.hidden_states[l].retain_grad()
                lg = model.lm_head(out.last_hidden_state[:, -1])
                if cap: lg = torch.tanh(lg / cap) * cap
                lg = lg.float(); ar = torch.arange(len(x), device=lg.device)
                d = lg[ar, b["io"][i:i+bs].to(lg.device)] - lg[ar, b["s"][i:i+bs].to(lg.device)]
                d.sum().backward()
            for l in layers: G[l].append(out.hidden_states[l].grad[:, -1].float().cpu())
            tid.append(torch.full((len(x),), t))
    return {l: torch.cat(v) for l, v in G.items()}, torch.cat(tid)

def center_by(X, tid):
    X = X.clone()
    for t in tid.unique():
        m = tid == t; X[m] = X[m] - X[m].mean(0)
    return X

def subspace(X, k, center=True):
    X = X.float(); X = X - X.mean(0) if center else X
    return torch.linalg.svd(X, full_matrices=False)[2][:k]

def angle(X, Y, k, center=True):
    s = torch.linalg.svdvals(subspace(X, k, center) @ subspace(Y, k, center).T).clamp(-1, 1)
    return torch.rad2deg(torch.arccos(s)).mean().item()

def shuf(Z, g): return Z[torch.randperm(len(Z), generator=g)]

In [8]:
def erase_hook(V, end_only):
    def fn(mod, args, out):
        tup = isinstance(out, tuple); h = out[0] if tup else out
        Vd = V.to(h.device)
        if end_only:
            h = h.clone(); last = h[:, -1].float()
            h[:, -1] = (last - (last @ Vd.T) @ Vd).to(h.dtype)
        else:
            hf = h.float(); h = (hf - (hf @ Vd.T) @ Vd).to(h.dtype)
        return (h,) + tuple(out[1:]) if tup else h
    return fn

def ev_erase(model, batches, Lh, V, end_only, NL):
    """erase subspace V (rows orthonormal) from residual output of layers Lh-1 .. NL-1."""
    with Hooks() as H:
        for l in range(Lh - 1, NL):
            H.add(model.model.layers[l], "register_forward_hook", erase_hook(V, end_only))
        return ev(model, batches)[0]

In [9]:
def run_model(name):
    path = f"{RESULTS_DIR}/{name.split('/')[-1]}{'_smoke' if SMOKE else ''}.json"
    if os.path.exists(path):
        print("cached", path); return json.load(open(path))
    tok = AutoTokenizer.from_pretrained(name, token=HF_TOKEN)
    model = AutoModelForCausalLM.from_pretrained(name, dtype=torch.bfloat16, device_map="auto",
                                                 attn_implementation="eager", token=HF_TOKEN).eval()
    for p in model.parameters(): p.requires_grad_(False)
    NL, NH = model.config.num_hidden_layers, model.config.num_attention_heads
    res = {"model": name, "gate": {}, "angles": [], "heads": {}, "subspace": {}, "ctrl": {}, "n_names": {}}

    names, nid = {}, {}
    for L in ("en", "fr", "yo"):
        names[L], nid[L] = single_token_names(tok, NAMES[L])
        random.Random(7).shuffle(names[L])
    nmin = min(len(v) for v in names.values())
    print("single-token names:", {k: len(v) for k, v in names.items()}, "-> using", nmin, "each")
    if nmin < 12: print("  WARNING few names")
    for L in names: names[L] = names[L][:nmin]
    res["n_names"] = nmin
    nid_all = {n: i for L in nid for n, i in nid[L].items()}
    nA, nB = names["en"][:nmin // 2], names["en"][nmin // 2:]
    T = list(range(6))
    SPEC = {"EN": ("en", names["en"]), "FR": ("fr", names["fr"]), "YO": ("yo", names["yo"]),
            "FRs": ("fr", names["en"]), "YOs": ("yo", names["en"]),
            "EN_NA": ("en", nA), "EN_NB": ("en", nB)}

    def conds(seed, n):
        r = random.Random(seed)
        return {k: build(tok, lang, T, nm, nid_all, n, r) for k, (lang, nm) in SPEC.items()}

    # 1. gate (estimation set, seed 0)
    C0 = conds(SEEDS[0], N_PROMPTS); ok = {}
    for k in ("EN", "FR", "YO", "FRs", "YOs"):
        ld, acc, _ = ev(model, C0[k])
        res["gate"][k] = {"acc": acc.mean().item(), "logit_diff": ld.mean().item(), "n": len(ld)}
        ok[k] = acc.mean().item() >= GATE_ACC
    print("GATE", {k: round(v["acc"], 3) for k, v in res["gate"].items()}, "ok:", ok)
    keys = ["EN", "EN_NA", "EN_NB"] + [k for k in ("FR", "YO", "FRs", "YOs") if ok[k]]
    MAT = [k for k in ("EN", "FR", "YO", "FRs", "YOs") if ok[k]]

    layers = [int(f * NL) for f in LAYER_FRACS]
    glayers = sorted(set(int(f * NL) for f in GRAD_FRACS))
    Lh = int(CAUSAL_FRAC * NL)
    BASE = {"FR": "FR", "FRs": "FR", "YO": "YO", "YOs": "YO"}

    # 2. geometry + stash per-seed gradient subspace sources and END residuals at Lh
    GRs, ENDs = {}, {}
    for seed in SEEDS:
        C = C0 if seed == SEEDS[0] else conds(seed, N_PROMPTS)
        g = torch.Generator().manual_seed(seed)
        H, TI, GR = {}, {}, {}
        for k in keys:
            H[k], TI[k] = ev_hs(model, C[k], layers)
            GR[k] = grad_END(model, C[k], glayers)[0]
        GRs[seed] = {k: GR[k][Lh] for k in keys}; ENDs[seed] = {k: H[k][Lh]["END"] for k in keys}

        def rows_for(getX, center, pos, frac):
            X = {k: shuf(getX(k), g) for k in keys}
            n = min(len(X[k]) // 2 for k in keys)
            row = {"seed": seed, "frac": frac, "pos": pos,
                   "floor_nameset": angle(X["EN_NA"][:n], X["EN_NB"][:n], K_ANGLE, center)}
            for L in ("EN", "FR", "YO"):
                if L in keys: row["floor_split_" + L] = angle(X[L][:n], X[L][n:2*n], K_ANGLE, center)
            for c in ("FR", "FRs", "YO", "YOs"):
                if c in keys:
                    cross = angle(X["EN"][:n], X[c][:n], K_ANGLE, center)
                    row["EN-" + c] = cross
                    row["excess_" + c] = cross - max(row["floor_split_EN"], row["floor_split_" + BASE[c]], row["floor_nameset"])
            res["angles"].append(row)

        for f, l in zip(LAYER_FRACS, layers):
            for p in ("IO", "S2", "END"):
                rows_for(lambda k: center_by(H[k][l][p], TI[k]), True, p, f)
        for f in GRAD_FRACS:
            l = int(f * NL)
            rows_for(lambda k: GR[k][l], False, "GRAD", f)
        del H, GR

    # held-out eval set for all causal tests
    EV = conds(999, N_EVAL)
    cleans = {k: ev(model, EV[k])[0].mean().item() for k in MAT}

    # 3a. heads: rank on estimation prompts per language, evaluate on held-out
    tops = {}
    for L, k in (("en", "EN"), ("fr", "FR"), ("yo", "YO")):
        if not ok[k]: continue
        sbs = [slice_b(b, SCAN_PER_T) for b in C0[k]]
        base = ev(model, sbs)[0].mean().item()
        sc = np.zeros((NL, NH))
        for l in range(NL):
            for h in range(NH):
                sc[l, h] = base - ev_heads(model, sbs, [(l, h)]).mean().item()
        flat = sorted(((sc[l, h], l, h) for l in range(NL) for h in range(NH)), reverse=True)
        tops[k] = [(l, h) for _, l, h in flat[:K_HEADS]]
    rr = random.Random(0); allh = [(l, h) for l in range(NL) for h in range(NH)]
    rands = [rr.sample(allh, K_HEADS) for _ in range(3)]
    res["heads"] = {"top": tops, "jaccard": {}, "retained": {}, "retained_rand": {}}
    for a in tops:
        for b in tops:
            if a < b: res["heads"]["jaccard"][f"{a}-{b}"] = len(set(tops[a]) & set(tops[b])) / len(set(tops[a]) | set(tops[b]))
    for tgt in MAT:
        res["heads"]["retained"][tgt] = {src: ev_heads(model, EV[tgt], tops[src]).mean().item() / cleans[tgt] for src in tops}
        res["heads"]["retained_rand"][tgt] = float(np.mean([ev_heads(model, EV[tgt], r).mean().item() for r in rands])) / cleans[tgt]

    # 3b. erasure of gradient subspaces (re-estimated per seed), held-out eval; random + PCA controls per target
    gg = torch.Generator().manual_seed(0)
    for m in ERASE_MODES:
        res["subspace"][m] = {K: {s: {t: [] for t in MAT} for s in MAT} for K in K_LIST}
        res["ctrl"][m] = {K: {"rand": {t: [] for t in MAT}, "pca": {t: [] for t in MAT}} for K in K_LIST}
    for seed in SEEDS:
        for K in K_LIST:
            Kc = min(K, min(len(GRs[seed][k]) for k in MAT))
            Dd = GRs[seed]["EN"].shape[1]
            Vr = torch.linalg.qr(torch.randn(Dd, Kc, generator=gg))[0].T
            Vs = {s: subspace(GRs[seed][s], Kc, center=False) for s in MAT}
            Vp = {t: subspace(ENDs[seed][t], Kc, center=True) for t in MAT}
            for m in ERASE_MODES:
                eo = m == "end"
                for t in MAT:
                    for s in MAT:
                        res["subspace"][m][K][s][t].append(ev_erase(model, EV[t], Lh, Vs[s], eo, NL).mean().item() / cleans[t])
                    res["ctrl"][m][K]["rand"][t].append(ev_erase(model, EV[t], Lh, Vr, eo, NL).mean().item() / cleans[t])
                    res["ctrl"][m][K]["pca"][t].append(ev_erase(model, EV[t], Lh, Vp[t], eo, NL).mean().item() / cleans[t])
        print("seed", seed, "done")

    json.dump(res, open(path, "w"), indent=1)
    del model; gc.collect(); torch.cuda.empty_cache()
    return res

In [10]:
def cross_ratio(mat, s, t):
    own = 1 - mat[t][t]
    return (1 - mat[s][t]) / own if own > OWN_MIN else float("nan")

SUMMARY = []
ALL = {m: run_model(m) for m in MODELS}

for m, r in ALL.items():
    print("\n=====", m, "| names per language:", r["n_names"])
    print("GATE:", {k: round(v["acc"], 3) for k, v in r["gate"].items()})
    df = pd.DataFrame(r["angles"])
    cols = [c for c in ["floor_split_EN", "floor_split_FR", "floor_split_YO", "floor_nameset", "EN-FR", "EN-FRs", "EN-YO", "EN-YOs",
                        "excess_FR", "excess_FRs", "excess_YO", "excess_YOs"] if c in df.columns]
    print("\nGEOMETRY (deg, matched halves; excess = cross - max(floors); FRs/YOs = same English names)")
    print(df.groupby(["frac", "pos"])[cols].agg(["mean", "std"]).round(1).to_string())
    h = r["heads"]
    print("\nHEAD ABLATION retained, held-out (rows=tgt, cols=heads ranked on src):")
    print(pd.DataFrame(h["retained"]).T.round(3).to_string())
    print("random heads:", {k: round(v, 3) for k, v in h["retained_rand"].items()}, " Jaccard:", {k: round(v, 2) for k, v in h["jaccard"].items()})
    for mode, byK in r["subspace"].items():
        for K, mat in byK.items():
            M = {s: {t: float(np.mean(v)) for t, v in d.items()} for s, d in mat.items()}
            S = {s: {t: float(np.std(v)) for t, v in d.items()} for s, d in mat.items()}
            ctrl = r["ctrl"][mode][K]
            print(f"\nERASURE mode={mode} K={K}: retained mean over seeds (rows=src subspace, cols=tgt); own effect > {OWN_MIN} = control OK")
            print(pd.DataFrame(M).T.round(3).to_string())
            print("  std:", {s: {t: round(x, 3) for t, x in d.items()} for s, d in S.items()} if len(SEEDS) > 1 else "(1 seed)")
            print("  random:", {t: round(float(np.mean(v)), 3) for t, v in ctrl["rand"].items()})
            print("  pca   :", {t: round(float(np.mean(v)), 3) for t, v in ctrl["pca"].items()})
            print("  own-OK:", {t: (1 - M[t][t]) > OWN_MIN for t in M})
            print("  cross/own:", {f"{s}->{t}": round(cross_ratio(M, s, t), 2) for s in M for t in M if s != t})
            for s, t in (("EN", "FR"), ("EN", "FRs"), ("EN", "YO"), ("EN", "YOs"), ("FR", "EN"), ("YO", "EN")):
                if s in M and t in M:
                    SUMMARY.append({"model": m.split("/")[-1], "mode": mode, "K": K, "pair": f"{s}->{t}",
                                    "ratio": cross_ratio(M, s, t), "retained": M[s][t], "random": float(np.mean(ctrl["rand"][t])),
                                    "pca": float(np.mean(ctrl["pca"][t]))})

print("\n\n##### SCALE SUMMARY (cross/own ratio; nan = target's own subspace failed its control) #####")
sdf = pd.DataFrame(SUMMARY)
if len(sdf):
    print(sdf.pivot_table(index=["mode", "K", "pair"], columns="model", values="ratio", sort=False).round(2).to_string())
    print("\nHEADS retained, EN-ranked heads -> tgt (random in brackets):")
    for m, r in ALL.items():
        print(m.split("/")[-1], {t: (round(v.get("EN", float("nan")), 3), round(r["heads"]["retained_rand"][t], 3)) for t, v in r["heads"]["retained"].items()})

Loading weights: 100%|██████████| 464/464 [00:12<00:00, 36.09it/s]


single-token names: {'en': 45, 'fr': 34, 'yo': 20} -> using 20 each
GATE {'EN': 1.0, 'FR': 1.0, 'YO': 0.995, 'FRs': 0.998, 'YOs': 0.998} ok: {'EN': True, 'FR': True, 'YO': True, 'FRs': True, 'YOs': True}
seed 0 done
seed 1 done
seed 2 done


Loading weights: 100%|██████████| 508/508 [00:37<00:00, 13.57it/s]


single-token names: {'en': 45, 'fr': 34, 'yo': 20} -> using 20 each
GATE {'EN': 0.998, 'FR': 1.0, 'YO': 1.0, 'FRs': 1.0, 'YOs': 1.0} ok: {'EN': True, 'FR': True, 'YO': True, 'FRs': True, 'YOs': True}
seed 0 done
seed 1 done
seed 2 done


Loading weights: 100%|██████████| 288/288 [00:06<00:00, 42.01it/s]


single-token names: {'en': 45, 'fr': 34, 'yo': 20} -> using 20 each
GATE {'EN': 1.0, 'FR': 0.988, 'YO': 0.867, 'FRs': 0.979, 'YOs': 0.983} ok: {'EN': True, 'FR': True, 'YO': True, 'FRs': True, 'YOs': True}
seed 0 done
seed 1 done
seed 2 done


Loading weights: 100%|██████████| 464/464 [00:23<00:00, 19.75it/s]


single-token names: {'en': 45, 'fr': 34, 'yo': 20} -> using 20 each
GATE {'EN': 0.998, 'FR': 0.998, 'YO': 0.986, 'FRs': 0.995, 'YOs': 0.986} ok: {'EN': True, 'FR': True, 'YO': True, 'FRs': True, 'YOs': True}
seed 0 done
seed 1 done
seed 2 done

===== /home/models/gemma-2-9b-it | names per language: 20
GATE: {'EN': 1.0, 'FR': 1.0, 'YO': 0.995, 'FRs': 0.998, 'YOs': 0.998}

GEOMETRY (deg, matched halves; excess = cross - max(floors); FRs/YOs = same English names)
          floor_split_EN      floor_split_FR      floor_split_YO       floor_nameset      EN-FR      EN-FRs      EN-YO      EN-YOs      excess_FR      excess_FRs      excess_YO      excess_YOs     
                    mean  std           mean  std           mean   std          mean  std  mean  std   mean  std  mean  std   mean  std      mean  std       mean  std      mean  std       mean  std
frac pos                                                                                                                                   